In [1]:
import sys
import os
sys.path.append(os.path.expanduser(r"C:\thesis\code\official_projects\otc-github\open-the-chests"))

## Dataset Mode for `Pattern` – Sampling from Pre-Recorded Traces

This notebook tests and demonstrates the **dataset mode** added to the `Pattern` class.

Previously, `Pattern` only supported *config mode*: events are generated stochastically from Allen-relation instructions at runtime. Dataset mode adds a second path where **pre-recorded event traces are loaded from a CSV file** and randomly sampled instead of generated.

### Two instruction formats — same list-of-dicts structure

The mode is selected by the presence of a `"dataset"` command in the instruction list:

| Command | Mode | Meaning |
|---|---|---|
| `"instantiate"`, Allen relations | Config | Generate events from a temporal logic spec |
| `"dataset"` | Dataset | Load traces from a CSV and sample at runtime |

**Config instruction example:**
```python
[
    {"command": "delay", "parameters": 10},
    {"command": "instantiate", "parameters": ("A", {"bg": "red"}, {"mu": 2, "sigma": 1}), "variable_name": "e1"},
]
```

**Dataset instruction example:**
```python
[
    {"command": "dataset", "parameters": "path/to/activity.csv"},
    {"command": "delay", "parameters": 10},   # optional
    {"command": "noise", "parameters": 0.0},  # optional
]
```

### CSV format

The CSV must follow the format produced by the data generation pipeline:

| Column | Description |
|---|---|
| `unique_activity_key` | Trace identifier (groups rows into one trace) |
| `device_id` | Event type (sensor name) |
| `start_time` | `HH:MM:SS.ffffff` – converted to seconds |
| `end_time` | `HH:MM:SS.ffffff` – converted to seconds |

Times are **normalized to the trace start** (first event starts at 0).

### Setup – Create a Synthetic CSV for Testing

Since we don't want to depend on external data files in this notebook, we generate a small synthetic CSV in a temporary file. It contains three traces of a fake `cook_dinner` activity with two sensor types.

In [2]:
import tempfile
import os

# Write a small synthetic per-activity CSV to a temp file
CSV_CONTENT = """unique_activity_key,device_id,start_time,end_time
file1_cook_dinner_1,floor_kitchen,00:00:01.000000,00:00:05.000000
file1_cook_dinner_1,tap_kitchen,00:00:03.000000,00:00:10.000000
file1_cook_dinner_1,oven,00:00:07.000000,00:00:20.000000
file2_cook_dinner_1,floor_kitchen,00:00:00.500000,00:00:04.000000
file2_cook_dinner_1,tap_kitchen,00:00:02.000000,00:00:08.000000
file2_cook_dinner_1,oven,00:00:06.000000,00:00:18.000000
file3_cook_dinner_1,floor_kitchen,00:00:00.000000,00:00:03.500000
file3_cook_dinner_1,tap_kitchen,00:00:02.500000,00:00:09.000000
file3_cook_dinner_1,door_fridge,00:00:08.000000,00:00:15.000000
file3_cook_dinner_1,oven,00:00:05.000000,00:00:17.000000
"""

# Write to a temp file
tmp = tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False)
tmp.write(CSV_CONTENT)
tmp.close()
DATA_FILE = tmp.name

print(f"Synthetic CSV written to: {DATA_FILE}")
print("\nCSV contents:")
print(CSV_CONTENT)

Synthetic CSV written to: C:\Users\Iva\AppData\Local\Temp\tmpqghvgu35.csv

CSV contents:
unique_activity_key,device_id,start_time,end_time
file1_cook_dinner_1,floor_kitchen,00:00:01.000000,00:00:05.000000
file1_cook_dinner_1,tap_kitchen,00:00:03.000000,00:00:10.000000
file1_cook_dinner_1,oven,00:00:07.000000,00:00:20.000000
file2_cook_dinner_1,floor_kitchen,00:00:00.500000,00:00:04.000000
file2_cook_dinner_1,tap_kitchen,00:00:02.000000,00:00:08.000000
file2_cook_dinner_1,oven,00:00:06.000000,00:00:18.000000
file3_cook_dinner_1,floor_kitchen,00:00:00.000000,00:00:03.500000
file3_cook_dinner_1,tap_kitchen,00:00:02.500000,00:00:09.000000
file3_cook_dinner_1,door_fridge,00:00:08.000000,00:00:15.000000
file3_cook_dinner_1,oven,00:00:05.000000,00:00:17.000000



### Test 1 – Creating a Dataset-Mode Pattern

Verify that `Pattern.__init__` correctly detects the `"dataset"` command and sets `instruction_type = "dataset"`.

In [3]:
from openthechests.src.elements.Pattern import Pattern

dataset_instruction = [
    {"command": "dataset",   "parameters": DATA_FILE},
    {"command": "activity",  "parameters": "cook_dinner"},
    {"command": "delay",     "parameters": 5.0},
    {"command": "noise",     "parameters": 0.0},
]

dp = Pattern(instruction=dataset_instruction, id=0)

print("instruction_type :", dp.instruction_type)
print("activity_name    :", dp.activity_name)
print("timeout          :", dp.timeout)
print("noise            :", dp.noise)
print("instruction      :", dp.instruction)   # should be []
print("num traces loaded:", len(dp.traces))

assert dp.instruction_type == "dataset"
assert dp.activity_name == "cook_dinner"
assert dp.timeout == 5.0
assert dp.instruction == []
assert len(dp.traces) == 3, f"Expected 3 traces, got {len(dp.traces)}"
print("\n✓ All assertions passed.")

instruction_type : dataset
activity_name    : cook_dinner
timeout          : 5.0
noise            : 0.0
instruction      : []
num traces loaded: 3

✓ All assertions passed.


### Test 2 – Inspect Loaded Traces

Check that `_load_traces` correctly parses the CSV: times are converted to seconds, normalized to trace start, and each trace is sorted by event end time.

In [4]:
for i, trace in enumerate(dp.traces):
    print(f"\nTrace {i} ({len(trace)} events):")
    for event in trace:
        print(f"  {event}")

# Verify: every trace starts at 0
for i, trace in enumerate(dp.traces):
    min_start = min(e.start for e in trace)
    assert min_start == 0.0, f"Trace {i} does not start at 0 (min start = {min_start})"

# Verify: each trace is sorted by end time
for i, trace in enumerate(dp.traces):
    ends = [e.end for e in trace]
    assert ends == sorted(ends), f"Trace {i} is not sorted by end time"

# Verify: events have empty attributes
for i, trace in enumerate(dp.traces):
    for event in trace:
        assert event.attributes == {}, f"Expected empty attributes, got {event.attributes}"

print("\n✓ All trace validation checks passed.")


Trace 0 (3 events):
  Event(type='floor_kitchen', attr={}, start=0.0, end=4.0)
  Event(type='tap_kitchen', attr={}, start=2.0, end=9.0)
  Event(type='oven', attr={}, start=6.0, end=19.0)

Trace 1 (3 events):
  Event(type='floor_kitchen', attr={}, start=0.0, end=3.5)
  Event(type='tap_kitchen', attr={}, start=1.5, end=7.5)
  Event(type='oven', attr={}, start=5.5, end=17.5)

Trace 2 (4 events):
  Event(type='floor_kitchen', attr={}, start=0.0, end=3.5)
  Event(type='tap_kitchen', attr={}, start=2.5, end=9.0)
  Event(type='door_fridge', attr={}, start=8.0, end=15.0)
  Event(type='oven', attr={}, start=5.0, end=17.0)

✓ All trace validation checks passed.


### Test 3 – `sample_timeout()` works the same as in config mode

In [5]:
import random
random.seed(42)

samples = [dp.sample_timeout() for _ in range(10)]
print("Sampled timeouts:", [round(s, 3) for s in samples])

assert all(0.0 <= s <= dp.timeout for s in samples), "Timeout samples out of range"
print("\n✓ All sampled timeouts are within [0, timeout].")

Sampled timeouts: [3.197, 0.125, 1.375, 1.116, 3.682, 3.383, 4.461, 0.435, 2.11, 0.149]

✓ All sampled timeouts are within [0, timeout].


### Test 4 – `visualize()` returns gracefully for dataset patterns

In [6]:
dp.visualize()  # Should print a message and return without error

Pattern 0 (cook_dinner) is a dataset pattern — no instruction graph to visualize.


### Test 5 – Config-mode Pattern still works unchanged

Make sure the existing config-mode path is unaffected by the change.

In [7]:
config_instruction = [
    {"command": "delay",  "parameters": 10},
    {"command": "noise",  "parameters": 0.1},
    {
        "command": "instantiate",
        "parameters": ("A", {"bg": "yellow", "fg": "red"}, {"mu": 2, "sigma": 1}),
        "variable_name": "e1"
    },
    {
        "command": "instantiate",
        "parameters": ("B", {"bg": "blue", "fg": "red"}, {"mu": 6, "sigma": 1}),
        "variable_name": "e2"
    },
    {
        "command": "after",
        "parameters": ["e2", "e1"],
        "variable_name": "e2",
        "other": {"gap_dist": {"mu": 4, "sigma": 1}}
    }
]

cp = Pattern(instruction=config_instruction, id=1)

print("instruction_type :", cp.instruction_type)
print("timeout          :", cp.timeout)
print("noise            :", cp.noise)
print("num instructions :", len(cp.instruction))

assert cp.instruction_type == "config"
assert cp.timeout == 10
assert cp.noise == 0.1
assert len(cp.instruction) == 3   # instantiate x2 + after
assert not hasattr(cp, "traces"), "Config pattern should not have a traces attribute"
print("\n✓ Config-mode pattern unaffected.")

instruction_type : config
timeout          : 10
noise            : 0.1
num instructions : 3

✓ Config-mode pattern unaffected.


### Test 6 – Dataset defaults (no delay / noise commands)

Confirm that omitting `"delay"` and `"noise"` falls back to 0.

In [8]:
minimal_instruction = [
    {"command": "dataset", "parameters": DATA_FILE},
]

mp = Pattern(instruction=minimal_instruction, id=2)

print("timeout (default)     :", mp.timeout)
print("noise   (default)     :", mp.noise)
print("activity_name (default):", mp.activity_name)

assert mp.timeout == 0
assert mp.noise == 0
assert mp.activity_name is None
print("\n✓ Defaults correct.")

timeout (default)     : 0
noise   (default)     : 0
activity_name (default): None

✓ Defaults correct.


### Test 6b – `activity` command sets `activity_name` in both modes

The `"activity"` command is optional in both dataset and config modes. It is consumed as a control command (filtered out of `self.instruction`) and stored on `self.activity_name` for readability and display.

In [9]:
# Dataset mode — with activity name
dp_named = Pattern(instruction=[
    {"command": "dataset",  "parameters": DATA_FILE},
    {"command": "activity", "parameters": "cook_dinner"},
    {"command": "delay",    "parameters": 3.0},
], id=3)

assert dp_named.activity_name == "cook_dinner"
assert dp_named.instruction == []          # activity filtered from instruction list
print("Dataset mode  — activity_name:", dp_named.activity_name)
dp_named.visualize()                       # should mention "cook_dinner" in message

# Config mode — with activity name
cp_named = Pattern(instruction=[
    {"command": "activity",    "parameters": "my_activity"},
    {"command": "delay",       "parameters": 5},
    {"command": "instantiate",
     "parameters": ("A", {"bg": "yellow", "fg": "red"}, {"mu": 2, "sigma": 0.5}),
     "variable_name": "e1"},
], id=4)

assert cp_named.activity_name == "my_activity"
assert all(cmd["command"] != "activity" for cmd in cp_named.instruction), \
    "'activity' must be filtered out of config instruction list"
print("Config mode   — activity_name:", cp_named.activity_name)
print("Config instructions remaining:", [c["command"] for c in cp_named.instruction])

print("\n✓ activity_name works correctly in both modes.")

Dataset mode  — activity_name: cook_dinner
Pattern 3 (cook_dinner) is a dataset pattern — no instruction graph to visualize.
Config mode   — activity_name: my_activity
Config instructions remaining: ['instantiate']

✓ activity_name works correctly in both modes.


---
## Parser – `instantiate_pattern(pattern)` with Dataset Mode

The key change to `Parser` is that `instantiate_pattern` now accepts a full `Pattern` object instead of a raw instruction list. It branches on `pattern.instruction_type`:

- **Config**: same as before — runs instantiate + Allen-relation commands
- **Dataset**: calls `random.choice(pattern.traces)`, records event durations so noise generation has valid statistics, and returns the sampled trace

Additional guards were added so that events with **empty attributes** (all dataset-mode events) don't crash `event_to_labelled` or `_check_event_values`.

### Test 7 – `instantiate_pattern` with a dataset-mode Pattern

Check that the sampled trace comes from the pre-loaded data, durations are recorded, and the result is sorted by end time.

In [10]:
import random
from openthechests.src.elements.Parser import Parser
from openthechests.src.elements.Pattern import Pattern

# The sensor names in the CSV are the event types
SENSOR_TYPES = ["floor_kitchen", "tap_kitchen", "oven", "door_fridge"]

parser = Parser(
    all_event_types=SENSOR_TYPES,
    all_noise_types=[],
    all_event_attributes={},
    all_noise_attributes={},
)

dp = Pattern(instruction=[{"command": "dataset", "parameters": DATA_FILE},
                           {"command": "activity", "parameters": "cook_dinner"},
                           {"command": "delay",   "parameters": 5.0}], id=0)

random.seed(0)
events = parser.instantiate_pattern(dp)

print(f"Sampled {len(events)} events:")
for e in events:
    print(f"  {e}")

# Result must match one of the loaded traces exactly
all_trace_sets = [frozenset((e.type, e.start, e.end) for e in trace) for trace in dp.traces]
result_set = frozenset((e.type, e.start, e.end) for e in events)
assert result_set in all_trace_sets, "Returned events do not match any loaded trace"

# Result must be sorted by end time
assert [e.end for e in events] == sorted(e.end for e in events), "Events not sorted by end time"

# Durations must have been recorded (default max was 1)
print(f"\nDuration stats after sampling: {parser.min_max_durations}")
assert parser.min_max_durations["max"] > 1, "Duration stats were not updated"

print("\n✓ instantiate_pattern (dataset mode) works correctly.")

Sampled 3 events:
  Event(type='floor_kitchen', attr={}, start=0.0, end=3.5)
  Event(type='tap_kitchen', attr={}, start=1.5, end=7.5)
  Event(type='oven', attr={}, start=5.5, end=17.5)

Duration stats after sampling: {'min': 1, 'max': np.float64(12.0)}

✓ instantiate_pattern (dataset mode) works correctly.


### Test 8 – `instantiate_pattern` config mode still works

Confirm the config path is untouched after the signature change.

In [12]:
config_parser = Parser(
    all_event_types=["A", "B"],
    all_noise_types=["N"],
    all_event_attributes={"bg": ["yellow", "blue"], "fg": ["red", "white"]},
    all_noise_attributes={"bg": ["gray"], "fg": ["black"]},
)

cp = Pattern(instruction=[
    {"command": "instantiate",
     "parameters": ("A", {"bg": "yellow", "fg": "red"}, {"mu": 2, "sigma": 0.5}),
     "variable_name": "e1"},
    {"command": "instantiate",
     "parameters": ("B", {"bg": "blue", "fg": "white"}, {"mu": 5, "sigma": 0.5}),
     "variable_name": "e2"},
    {"command": "after",
     "parameters": ["e2", "e1"],
     "variable_name": "e2",
     "other": {"gap_dist": {"mu": 3, "sigma": 0.5}}},
], id=1)

random.seed(1)
events = config_parser.instantiate_pattern(cp)

print(f"Generated {len(events)} events:")
for e in events:
    print(f"  {e}")

assert len(events) == 2
assert all(e.attributes != {} for e in events), "Config events should have attributes"
assert events[0].end <= events[1].end, "Events not sorted"
print("\n✓ instantiate_pattern (config mode) unchanged.")

Generated 2 events:
  Event(type='A', attr={'bg': 'yellow', 'fg': 'red'}, start=0, end=2.304)
  Event(type='B', attr={'bg': 'blue', 'fg': 'white'}, start=5.804, end=10.797)

✓ instantiate_pattern (config mode) unchanged.


### Test 9 – `event_to_labelled` with empty attributes

Dataset events carry no attributes. The updated guard should return an event with an empty attribute dict instead of crashing.

In [13]:
from openthechests.src.elements.Event import Event

data_event = Event(e_type="floor_kitchen", e_attributes={}, t_start=0.0, t_end=4.0)

labelled = parser.event_to_labelled(data_event)

print("Original event :", data_event)
print("Labelled event :", labelled)
print("Labelled type  :", labelled.type, " (index of 'floor_kitchen' in all_types)")
print("Attributes     :", labelled.attributes, " (empty — no encoding attempted)")

assert labelled.attributes == {}
assert labelled.type == SENSOR_TYPES.index("floor_kitchen")
print("\n✓ event_to_labelled handles empty attributes correctly.")

Original event : Event(type='floor_kitchen', attr={}, start=0.0, end=4.0)
Labelled event : Event(type='0', attr={}, start=0.0, end=4.0)
Labelled type  : 0  (index of 'floor_kitchen' in all_types)
Attributes     : {}  (empty — no encoding attempted)

✓ event_to_labelled handles empty attributes correctly.


---
## Generator – Full Event Loop with Dataset-Mode Patterns

The only change to `Generator` is a one-line update in `_fill_event_stack`:

```python
# Before
generated_events = self.parser.instantiate_pattern(pattern.instruction)
# After
generated_events = self.parser.instantiate_pattern(pattern)
```

Everything else — timeout offset, noise injection, sorting, refilling stacks — is inherited unchanged and works for both modes.

### Test 10 – Generator reset and event loop (two dataset patterns)

In [14]:
from openthechests.src.elements.Generator import Generator

random.seed(42)

p0 = Pattern(instruction=[{"command": "dataset", "parameters": DATA_FILE},
                           {"command": "delay",   "parameters": 3.0}], id=0)
p1 = Pattern(instruction=[{"command": "dataset", "parameters": DATA_FILE},
                           {"command": "delay",   "parameters": 2.0}], id=1)

gen_parser = Parser(
    all_event_types=SENSOR_TYPES,
    all_noise_types=[],
    all_event_attributes={},
    all_noise_attributes={},
)

gen = Generator(parser=gen_parser, patterns=[p0, p1], verbose=True)
gen.reset()

print("\n--- Processing events ---")
all_events = []
all_signals = []

for _ in range(12):
    event, signal = gen.next_event()
    all_events.append(event)
    all_signals.append(signal)
    print(f"  {event}  →  signals: {dict(signal)}")

# Events must be in non-decreasing end-time order
ends = [e.end for e in all_events]
assert ends == sorted(ends), "Events are not in chronological order"

# All event types must come from the sensor list
assert all(e.type in SENSOR_TYPES for e in all_events), "Unknown event type encountered"

print("\n✓ Generator produces chronological, valid events from dataset patterns.")

Sampling new events from pattern 0: [Event(type='floor_kitchen', attr={}, start=1.918, end=5.918), Event(type='tap_kitchen', attr={}, start=3.918, end=10.918), Event(type='oven', attr={}, start=7.918, end=20.918)]
Sampling new events from pattern 1: [Event(type='floor_kitchen', attr={}, start=1.483, end=5.483), Event(type='tap_kitchen', attr={}, start=3.483, end=10.483), Event(type='oven', attr={}, start=7.483, end=20.483)]

--- Processing events ---
  Event(type='floor_kitchen', attr={}, start=1.483, end=5.483)  →  signals: {1: ['active'], 0: ['active']}
  Event(type='floor_kitchen', attr={}, start=1.918, end=5.918)  →  signals: {0: ['active'], 1: ['active']}
  Event(type='tap_kitchen', attr={}, start=3.483, end=10.483)  →  signals: {1: ['active'], 0: ['active']}
  Event(type='tap_kitchen', attr={}, start=3.918, end=10.918)  →  signals: {0: ['active'], 1: ['active']}
Sampling new events from pattern 1: [Event(type='floor_kitchen', attr={}, start=20.93, end=24.43), Event(type='tap_kitc

### Test 11 – Timing offset is applied

With `delay > 0`, the first event in each stack after `reset()` must start after time 0 — shifted by the sampled timeout.

In [15]:
random.seed(7)

p_delay = Pattern(instruction=[{"command": "dataset", "parameters": DATA_FILE},
                                {"command": "delay",   "parameters": 10.0}], id=0)

gen_delay = Generator(parser=gen_parser, patterns=[p_delay], verbose=False)
gen_delay.reset()

first_event = gen_delay.event_stacks[0][0]
print("First event in stack after reset:", first_event)
print("Start time:", round(first_event.start, 4))

# Raw trace starts at 0; with delay=10 the offset is Uniform(0,10), so start >= 0
assert first_event.start >= 0.0

# Also check that a zero-delay pattern starts at 0
p_nodelay = Pattern(instruction=[{"command": "dataset", "parameters": DATA_FILE}], id=0)
gen_nodelay = Generator(parser=gen_parser, patterns=[p_nodelay], verbose=False)
random.seed(0)
gen_nodelay.reset()
first_nodelay = gen_nodelay.event_stacks[0][0]
print("First event (no delay):", first_nodelay)
assert first_nodelay.start == 0.0

print("\n✓ Timing offset correctly shifts trace events.")

First event in stack after reset: Event(type='floor_kitchen', attr={}, start=3.238, end=7.238)
Start time: 3.2383
First event (no delay): Event(type='floor_kitchen', attr={}, start=0.0, end=3.5)

✓ Timing offset correctly shifts trace events.


### Test 12 – Mixed mode: dataset pattern + config pattern in the same Generator

In [16]:
random.seed(99)

# Dataset pattern (id=0) — samples from CSV
p_data = Pattern(instruction=[{"command": "dataset", "parameters": DATA_FILE}], id=0)

# Config pattern (id=1) — generates events from Allen relations
p_cfg = Pattern(instruction=[
    {"command": "instantiate",
     "parameters": ("A", {"bg": "yellow", "fg": "red"}, {"mu": 3, "sigma": 0.5}),
     "variable_name": "e1"},
    {"command": "instantiate",
     "parameters": ("B", {"bg": "blue", "fg": "white"}, {"mu": 4, "sigma": 0.5}),
     "variable_name": "e2"},
    {"command": "after",
     "parameters": ["e2", "e1"],
     "variable_name": "e2",
     "other": {"gap_dist": {"mu": 2, "sigma": 0.5}}},
], id=1)

mixed_parser = Parser(
    all_event_types=SENSOR_TYPES + ["A", "B"],
    all_noise_types=[],
    all_event_attributes={"bg": ["yellow", "blue"], "fg": ["red", "white"]},
    all_noise_attributes={},
)

gen_mixed = Generator(parser=mixed_parser, patterns=[p_data, p_cfg], verbose=False)
gen_mixed.reset()

print("--- Mixed-mode event loop ---")
seen_types = set()
for _ in range(10):
    event, signal = gen_mixed.next_event()
    seen_types.add(event.type)
    print(f"  {event}  →  {dict(signal)}")

has_sensor = any(t in SENSOR_TYPES for t in seen_types)
has_config = any(t in ["A", "B"] for t in seen_types)
assert has_sensor, "No sensor events from dataset pattern"
assert has_config, "No A/B events from config pattern"

print(f"\nEvent types seen: {seen_types}")
print("✓ Mixed-mode generator produces events from both patterns.")

--- Mixed-mode event loop ---
  Event(type='A', attr={'bg': 'yellow', 'fg': 'red'}, start=0.0, end=2.733)  →  {1: ['active'], 0: ['active']}
  Event(type='floor_kitchen', attr={}, start=0.0, end=4.0)  →  {0: ['active']}
  Event(type='B', attr={'bg': 'blue', 'fg': 'white'}, start=4.233, end=8.321)  →  {1: ['active', 'satisfied'], 0: ['active']}
  Event(type='tap_kitchen', attr={}, start=2.0, end=9.0)  →  {0: ['active'], 1: ['active']}
  Event(type='A', attr={'bg': 'yellow', 'fg': 'red'}, start=8.321, end=11.784)  →  {1: ['active'], 0: ['active']}
  Event(type='B', attr={'bg': 'blue', 'fg': 'white'}, start=13.327, end=17.827)  →  {1: ['active', 'satisfied'], 0: ['active']}
  Event(type='oven', attr={}, start=6.0, end=19.0)  →  {0: ['active', 'satisfied'], 1: ['active']}
  Event(type='A', attr={'bg': 'yellow', 'fg': 'red'}, start=17.827, end=20.889)  →  {1: ['active'], 0: ['active']}
  Event(type='floor_kitchen', attr={}, start=19.0, end=22.5)  →  {0: ['active'], 1: ['active']}
  Event(ty

### Cleanup

In [17]:
os.unlink(DATA_FILE)
print("Temp file removed.")

Temp file removed.
